In [1]:
# ===============================
# 📦 IMPORTS
# ===============================
import os
import pandas as pd
import numpy as np
import joblib
import copy

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, r2_score, mean_squared_log_error

In [2]:
import mlflow

mlflow.set_tracking_uri("file:f:/streamlit_session/guvi_projects/smart_premium/mlruns")

mlflow.set_experiment("Insurance_Premium_Project")

print("✅ CLEAN FILE-BASED MLflow ACTIVE")

f:\streamlit_session\mlenv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
Traceback (most recent call last):
  File "f:\streamlit_session\mlenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f:\streamlit_session\mlenv\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "f

✅ CLEAN FILE-BASED MLflow ACTIVE


In [3]:

# ===============================
# 📊 LOAD DATA
# ===============================
df = pd.read_csv("data/processed/train_cleaned.csv")




# ✅ NO LOG TRANSFORMATION
X = df.drop(columns=["Premium Amount"], errors="ignore")
y = df["Premium Amount"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [4]:
# ===============================
# 🔧 PREPROCESSING (FIXED)
# ===============================
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])



In [5]:
# ===============================
# 🤖 MODELS
# ===============================
models = {
    "LinearRegression": LinearRegression(),

    "RandomForest": RandomForestRegressor(
        n_estimators=150,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=150,
        learning_rate=0.08,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1,
        n_jobs=-1,
        random_state=42,
        verbosity=1
    )
}



In [ ]:

# ===============================
# 🏋️ TRAINING LOOP
# ===============================
results = {}

best_model = None
best_score = float("inf")
best_model_name = None

for name, model in models.items():

    with mlflow.start_run(run_name=name):

        print(f"\n🚀 Training {name}...")

        # ✅ Create NEW preprocessor each time
        preprocessor_new = ColumnTransformer([
            ("num", num_pipeline, num_cols),
            ("cat", cat_pipeline, cat_cols)
        ])

        # ✅ Fresh pipeline (IMPORTANT FIX)
        pipe = Pipeline([
            ("preprocessor", preprocessor_new),
            ("model", model)   # ✅ NO deepcopy needed
        ])

        # ✅ Train
        pipe.fit(X_train, y_train)

        # ✅ Predict
        y_pred = pipe.predict(X_test)

        # ✅ Metrics
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        try:
            rmsle = np.sqrt(mean_squared_log_error(y_test, y_pred))
        except:
            rmsle = np.nan

        # ✅ Log to MLflow
        mlflow.log_param("model_name", name)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("rmsle", rmsle)

        print(f"✅ {name} logged")
        print(f"{name} → RMSE: {rmse:.4f}, R2: {r2:.4f}, RMSLE: {rmsle:.4f}")

        results[name] = (rmse, r2, rmsle)

    # ✅ BEST MODEL SELECTION
    if best_model is None or rmse < best_score:
        best_score = rmse
        best_model = pipe
        best_model_name = name


# ===============================
# ✅ SAVE FINAL PIPELINE ONLY
# ===============================

os.makedirs("model", exist_ok=True)

# ✅ SAVE FULL PIPELINE (THIS IS THE ONLY THING YOU NEED)
joblib.dump(best_model, "model/final_pipeline.pkl")

print("✅ FINAL PIPELINE SAVED")
print("Best Model:", best_model_name)



🚀 Training LinearRegression...
✅ LinearRegression logged
LinearRegression → RMSE: 863.2731, R2: 0.0027, RMSLE: 1.1687

🚀 Training RandomForest...
✅ RandomForest logged
RandomForest → RMSE: 851.2306, R2: 0.0304, RMSLE: 1.1598

🚀 Training XGBoost...
✅ XGBoost logged
XGBoost → RMSE: 845.4795, R2: 0.0434, RMSLE: 1.1482
✅ FINAL MODEL SAVED


In [7]:






# ===============================
# 📊 SAVE RESULTS CSV (ADD HERE ✅)
# ===============================
results_df = pd.DataFrame(results).T
results_df.columns = ["RMSE", "R2", "RMSLE"]
results_df.to_csv("model/model_results.csv")

print("\n✅ Results saved to model/model_results.csv")



# ===============================
# 📊 FINAL COMPARISON
# ===============================
print("\n📊 Final Model Comparison:\n")

results_df = pd.DataFrame(results).T
results_df.columns = ["RMSE", "R2", "RMSLE"]

print(results_df.sort_values("RMSLE"))



✅ Results saved to model/model_results.csv

📊 Final Model Comparison:

                        RMSE        R2     RMSLE
XGBoost           845.479532  0.043424  1.148157
RandomForest      851.230630  0.030366  1.159822
LinearRegression  863.273105  0.002737  1.168678
